In [2]:
from active_semi_clustering.semi_supervised.pairwise_constraints import PCKMeans

# 1. Data load

In [3]:
import os
import json
import time
import random
import ast
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances
from active_semi_clustering.semi_supervised.pairwise_constraints import PCKMeans

from dotenv import load_dotenv


load_dotenv()


api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

train_path = "/workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT/step0_TACRED_full_phrase.csv"
test_path = "/workspace/KG-known0/ARR2026/Data_paper/TACRED/Test/TACRED_test_phrase.csv"

save_dir = "/workspace/KG-known0/ARR2026/Baselines/4_FewShotClus/output_TACRED"
os.makedirs(save_dir, exist_ok=True)

df_train = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train shape:", df_train.shape)
print("test shape :", test_df.shape)

required_cols = ["text", "subject", "object", "gpt4_phrase", "relation"]

for col in required_cols:
    assert col in df_train.columns, f"df_train에 {col} 열이 없습니다."
    assert col in test_df.columns, f"test_df에 {col} 열이 없습니다."

random_state = 42
np.random.seed(random_state)
random.seed(random_state)

llm_model = "gpt-4.1-mini"
max_parallel_api_calls = 20

k0 = 300
num_constraints = len(df_train)

num_representatives = 3
num_candidate_clusters = 5

low_confidence_ratio = 0.15
num_low_confidence = max(1, int(round(len(test_df) * low_confidence_ratio)))

print("num_constraints:", num_constraints)
print("max_parallel_api_calls:", max_parallel_api_calls)
print("num_representatives:", num_representatives)
print("num_candidate_clusters:", num_candidate_clusters)
print("low_confidence_ratio:", low_confidence_ratio)
print("num_low_confidence:", num_low_confidence)


train shape: (1900, 20)
test shape : (760, 20)
num_constraints: 1900
max_parallel_api_calls: 20
num_representatives: 3
num_candidate_clusters: 5
low_confidence_ratio: 0.15
num_low_confidence: 114


# 1. relational phrase를 key로 임베딩 확장

In [4]:

def build_instance_text(df: pd.DataFrame, desc="build instance texts") -> list:
    texts = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc=desc):
        sent = str(row.text) if pd.notna(row.text) else ""
        subj = str(row.subject) if pd.notna(row.subject) else ""
        obj = str(row.object) if pd.notna(row.object) else ""

        instance = (
            f"Sentence: {sent}\n"
            f"Head entity: {subj}\n"
            f"Tail entity: {obj}"
        )
        texts.append(instance)

    return texts


train_instance_texts = build_instance_text(df_train, desc="build train instance texts")
test_instance_texts = build_instance_text(test_df, desc="build test instance texts")

train_keyphrase_texts = df_train["gpt4_phrase"].fillna("").astype(str).tolist()
test_keyphrase_texts = test_df["gpt4_phrase"].fillna("").astype(str).tolist()

print("train instance texts:", len(train_instance_texts))
print("train keyphrase texts:", len(train_keyphrase_texts))
print("test instance texts :", len(test_instance_texts))
print("test keyphrase texts:", len(test_keyphrase_texts))

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def encode_texts(texts, model, batch_size=64):
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings


print("\n[Embedding] train instance")
E_train_instance = encode_texts(train_instance_texts, model)

print("\n[Embedding] train gpt4_phrase")
E_train_keyphrase = encode_texts(train_keyphrase_texts, model)

print("\n[Embedding] test instance")
E_test_instance = encode_texts(test_instance_texts, model)

print("\n[Embedding] test gpt4_phrase")
E_test_keyphrase = encode_texts(test_keyphrase_texts, model)

with tqdm(total=3, desc="combine and normalize embeddings") as pbar:
    E_train_final = np.concatenate([E_train_instance, E_train_keyphrase], axis=1)
    E_test_final = np.concatenate([E_test_instance, E_test_keyphrase], axis=1)
    pbar.update(1)

    E_train_final = normalize(E_train_final, norm="l2")
    pbar.update(1)

    E_test_final = normalize(E_test_final, norm="l2")
    pbar.update(1)

print("E_train_final:", E_train_final.shape)
print("E_test_final :", E_test_final.shape)


build train instance texts:   0%|          | 0/1900 [00:00<?, ?it/s]

build test instance texts:   0%|          | 0/760 [00:00<?, ?it/s]

train instance texts: 1900
train keyphrase texts: 1900
test instance texts : 760
test keyphrase texts: 760

[Embedding] train instance


Batches:   0%|          | 0/30 [00:00<?, ?it/s]


[Embedding] train gpt4_phrase


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

... (2 outputs left out)


[Embedding] test gpt4_phrase


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

combine and normalize embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

E_train_final: (1900, 768)
E_test_final : (760, 768)


# During clustering: LLM pairwise constraint 생성 + PCKMeans

unknown k 추정

In [5]:
import numpy as np

print(np.geterr())

{'divide': 'raise', 'over': 'raise', 'under': 'raise', 'invalid': 'raise'}


In [6]:
np.seterr(
    divide="warn",
    over="warn",
    under="ignore",
    invalid="warn"
)

{'divide': 'raise', 'over': 'raise', 'under': 'raise', 'invalid': 'raise'}

In [7]:

def estimate_k_by_zhang2021(embeddings, k0, random_state=42):
    n = len(embeddings)

    if k0 >= n:
        raise ValueError(f"k0={k0}는 sample 수 n={n}보다 작아야 합니다.")

    with tqdm(total=3, desc="estimate K by Zhang2021") as pbar:
        kmeans_k0 = KMeans(
            n_clusters=k0,
            random_state=random_state,
            n_init="auto"
        )
        pbar.update(1)

        labels_k0 = kmeans_k0.fit_predict(embeddings)
        pbar.update(1)

        cluster_sizes = pd.Series(labels_k0).value_counts().sort_index()
        threshold = n / k0

        valid_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
        estimated_k = len(valid_clusters)

        cluster_info = pd.DataFrame({
            "cluster_id": cluster_sizes.index,
            "size": cluster_sizes.values,
            "threshold": threshold,
            "is_valid": cluster_sizes.values >= threshold
        })
        pbar.update(1)

    if estimated_k <= 1:
        raise ValueError(f"Estimated K={estimated_k}입니다. k0를 확인하세요.")

    return estimated_k, labels_k0, cluster_info


estimated_k, labels_k0, cluster_info = estimate_k_by_zhang2021(
    embeddings=E_train_final,
    k0=k0,
    random_state=random_state
)

print("Estimated K:", estimated_k)
display(cluster_info.head())
print(cluster_info["is_valid"].value_counts())


estimate K by Zhang2021:   0%|          | 0/3 [00:00<?, ?it/s]

Estimated K: 109


,cluster_id,size,threshold,is_valid
0,0,6,6.333333,False
1,1,8,6.333333,True
2,2,5,6.333333,False
3,3,5,6.333333,False
4,4,7,6.333333,True


is_valid
False    191
True     109
Name: count, dtype: int64


num_constraints개 pair 샘플링 (LLM이 두 샘플이 같은 클러스터에 속해야 하는지 판단)

In [8]:

def sample_random_pairs(n, num_pairs, random_state=42):
    rng = np.random.default_rng(random_state)
    pairs = set()

    max_possible = n * (n - 1) // 2
    if num_pairs > max_possible:
        raise ValueError(f"num_pairs={num_pairs}가 가능한 pair 수 {max_possible}보다 큽니다.")

    with tqdm(total=num_pairs, desc="sample random pairs") as pbar:
        while len(pairs) < num_pairs:
            before = len(pairs)
            i = int(rng.integers(0, n))
            j = int(rng.integers(0, n))

            if i == j:
                continue

            a, b = min(i, j), max(i, j)
            pairs.add((a, b))

            if len(pairs) > before:
                pbar.update(1)

    return list(pairs)


pairs = sample_random_pairs(
    n=len(df_train),
    num_pairs=num_constraints,
    random_state=random_state
)

print("sampled pairs:", len(pairs))


sample random pairs:   0%|          | 0/1900 [00:00<?, ?it/s]

sampled pairs: 1900


In [9]:

def build_pairwise_prompt(row_i, row_j):
    text_i = str(row_i["text"]) if pd.notna(row_i["text"]) else ""
    subj_i = str(row_i["subject"]) if pd.notna(row_i["subject"]) else ""
    obj_i = str(row_i["object"]) if pd.notna(row_i["object"]) else ""

    text_j = str(row_j["text"]) if pd.notna(row_j["text"]) else ""
    subj_j = str(row_j["subject"]) if pd.notna(row_j["subject"]) else ""
    obj_j = str(row_j["object"]) if pd.notna(row_j["object"]) else ""

    prompt = f"""
You are helping with relation-instance clustering.

Given two relation instances, determine whether they express the same semantic relation between the head entity and the tail entity.

Answer only one of the following:
- Must-link
- Cannot-link

Use "Must-link" if the two instances should belong to the same relation cluster.
Use "Cannot-link" if the two instances should belong to different relation clusters.

Instance 1:
Sentence: {text_i}
Head entity: {subj_i}
Tail entity: {obj_i}

Instance 2:
Sentence: {text_j}
Head entity: {subj_j}
Tail entity: {obj_j}

Answer:
""".strip()

    return prompt


def parse_constraint_output(output_text):
    text = output_text.strip().lower()

    if "cannot-link" in text or "cannot link" in text or "cannot" in text:
        return "cannot-link"

    if "must-link" in text or "must link" in text or "must" in text:
        return "must-link"

    return "error"


def call_llm_pairwise_constraint(row_i, row_j, model_name="gpt-4.1-mini", max_retries=5, sleep_sec=2):
    prompt = build_pairwise_prompt(row_i, row_j)

    for attempt in range(max_retries):
        try:
            result = client.responses.create(
                model=model_name,
                input=prompt,
                temperature=0
            )

            output_text = result.output_text
            parsed = parse_constraint_output(output_text)

            return parsed, output_text

        except Exception as e:
            tqdm.write(f"[ERROR] pairwise LLM attempt={attempt + 1}, error={e}")
            time.sleep(sleep_sec)

    return "error", "ERROR"


def run_pairwise_constraint_job(pair_idx, i, j):
    row_i = df_train.iloc[i]
    row_j = df_train.iloc[j]

    label, raw_output = call_llm_pairwise_constraint(
        row_i=row_i,
        row_j=row_j,
        model_name=llm_model
    )

    return pair_idx, {
        "pair_id": pair_idx,
        "i": i,
        "j": j,
        "llm_constraint": label,
        "raw_output": raw_output
    }


constraint_cache_path = os.path.join(
    save_dir,
    f"TACRED_train_llm_pairwise_constraints_{num_constraints}.csv"
)

if os.path.exists(constraint_cache_path):
    constraint_df = pd.read_csv(constraint_cache_path)
    print("[LOAD]", constraint_cache_path)

else:
    records_by_pair_id = {}
    counts = {"must-link": 0, "cannot-link": 0, "error": 0}

    with ThreadPoolExecutor(max_workers=max_parallel_api_calls) as executor:
        futures = [
            executor.submit(run_pairwise_constraint_job, pair_idx, i, j)
            for pair_idx, (i, j) in enumerate(pairs)
        ]

        progress = tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"API pairwise constraints x{max_parallel_api_calls}"
        )
        for done_count, future in enumerate(progress, start=1):
            pair_idx, record = future.result()
            label = record["llm_constraint"]
            counts[label] = counts.get(label, 0) + 1
            records_by_pair_id[pair_idx] = record

            progress.set_postfix({
                "must": counts.get("must-link", 0),
                "cannot": counts.get("cannot-link", 0),
                "error": counts.get("error", 0),
            })

            if done_count % 50 == 0:
                ordered_records = [
                    records_by_pair_id[k]
                    for k in sorted(records_by_pair_id)
                ]
                constraint_df = pd.DataFrame(ordered_records)
                constraint_df.to_csv(constraint_cache_path, index=False)
                tqdm.write(f"[SAVE] {done_count}/{len(pairs)}")

    ordered_records = [
        records_by_pair_id[k]
        for k in sorted(records_by_pair_id)
    ]
    constraint_df = pd.DataFrame(ordered_records)
    constraint_df.to_csv(constraint_cache_path, index=False)
    print("[DONE]", constraint_cache_path)


print(constraint_df["llm_constraint"].value_counts(dropna=False))

must_link = [
    (int(row["i"]), int(row["j"]))
    for _, row in tqdm(
        constraint_df.iterrows(),
        total=len(constraint_df),
        desc="build must-link constraints"
    )
    if row["llm_constraint"] == "must-link"
]

cannot_link = [
    (int(row["i"]), int(row["j"]))
    for _, row in tqdm(
        constraint_df.iterrows(),
        total=len(constraint_df),
        desc="build cannot-link constraints"
    )
    if row["llm_constraint"] == "cannot-link"
]

print("must_link:", len(must_link))
print("cannot_link:", len(cannot_link))

if len(must_link) + len(cannot_link) == 0:
    raise ValueError("유효한 LLM pairwise constraint가 없습니다.")


API pairwise constraints x20:   0%|          | 0/1900 [00:00<?, ?it/s]

[SAVE] 50/1900
[SAVE] 100/1900
[SAVE] 150/1900
[SAVE] 200/1900
[SAVE] 250/1900
[SAVE] 300/1900
[SAVE] 350/1900
[SAVE] 400/1900
[SAVE] 450/1900
[SAVE] 500/1900
[SAVE] 550/1900
[SAVE] 600/1900
[SAVE] 650/1900
[SAVE] 700/1900
[SAVE] 750/1900
[SAVE] 800/1900
[SAVE] 850/1900
[SAVE] 900/1900
[SAVE] 950/1900
[SAVE] 1000/1900
[SAVE] 1050/1900
[SAVE] 1100/1900
[SAVE] 1150/1900
[SAVE] 1200/1900
[SAVE] 1250/1900
[SAVE] 1300/1900
[SAVE] 1350/1900
[SAVE] 1400/1900
[SAVE] 1450/1900
[SAVE] 1500/1900
[SAVE] 1550/1900
[SAVE] 1600/1900
[SAVE] 1650/1900
[SAVE] 1700/1900
[SAVE] 1750/1900
[SAVE] 1800/1900
[SAVE] 1850/1900
[SAVE] 1900/1900
[DONE] /workspace/KG-known0/ARR2026/Baselines/4_FewShotClus/output_TACRED/TACRED_train_llm_pairwise_constraints_1900.csv
llm_constraint
cannot-link    1851
must-link        49
Name: count, dtype: int64


build must-link constraints:   0%|          | 0/1900 [00:00<?, ?it/s]

build cannot-link constraints:   0%|          | 0/1900 [00:00<?, ?it/s]

must_link: 49
cannot_link: 1851


PCKMeans

In [11]:

clusterer = PCKMeans(
    n_clusters=estimated_k,
   
)

with tqdm(total=1, desc="PCKMeans fit") as pbar:
    clusterer.fit(
        E_train_final,
        ml=must_link,
        cl=cannot_link
    )
    pbar.update(1)

df_train["cluster_after_pckmeans"] = clusterer.labels_

print("train cluster count:", df_train["cluster_after_pckmeans"].nunique())


PCKMeans fit:   0%|          | 0/1 [00:00<?, ?it/s]

train cluster count: 109


In [12]:

def compute_cluster_centers(embeddings, labels, n_clusters):
    centers = []

    for cid in tqdm(range(n_clusters), desc="compute cluster centers"):
        idx = np.where(labels == cid)[0]

        if len(idx) == 0:
            centers.append(np.zeros(embeddings.shape[1]))
        else:
            centers.append(embeddings[idx].mean(axis=0))

    centers = np.vstack(centers)
    centers = normalize(centers, norm="l2")

    return centers


train_labels = df_train["cluster_after_pckmeans"].to_numpy()
centers = compute_cluster_centers(
    embeddings=E_train_final,
    labels=train_labels,
    n_clusters=estimated_k
)

print("centers shape:", centers.shape)


compute cluster centers:   0%|          | 0/109 [00:00<?, ?it/s]

centers shape: (109, 768)


test set assignment

In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment


def evaluate_clustering(df: pd.DataFrame, pred_col: str, gold_col: str = "relation"):

    
    non_na_df = df[~df[pred_col].isna()].copy()

    if len(non_na_df) == 0:
        raise ValueError(f"{pred_col}에 유효한 값이 없습니다.")

    y_true = non_na_df[gold_col].astype(str).values
    y_pred = non_na_df[pred_col].astype(str).values

    
    overall_purity = (
        non_na_df
        .groupby(pred_col)[gold_col]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(non_na_df)
    )

    
    ari = adjusted_rand_score(y_true, y_pred)

    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)

    
    def b3_precision_recall_f1(y_true, y_pred):

        true_groups = {}
        pred_groups = {}

        for i, (t, p) in tqdm(
            enumerate(zip(y_true, y_pred)),
            total=len(y_true),
            desc=f"{pred_col} B3 group index"
        ):
            true_groups.setdefault(t, set()).add(i)
            pred_groups.setdefault(p, set()).add(i)

        P_sum, R_sum = 0.0, 0.0
        n = len(y_true)

        for i, (t, p) in tqdm(
            enumerate(zip(y_true, y_pred)),
            total=len(y_true),
            desc=f"{pred_col} B3 score"
        ):
            T = true_groups[t]
            C = pred_groups[p]
            inter = len(T & C)

            P_sum += inter / len(C) if len(C) > 0 else 0.0
            R_sum += inter / len(T) if len(T) > 0 else 0.0

        P = P_sum / n
        R = R_sum / n
        F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0

        return P, R, F1

    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    
    print(f"Purity : {overall_purity:.4f}")
    print(f"ACC    : {hungarian_acc:.4f}")
    
    
    print(f"B3-F1  : {b3_f1:.4f}")
    
    
    print(f"V-F1   : {v_f1:.4f}")
    print(f"ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }


In [14]:

with tqdm(total=1, desc="assign test clusters by nearest center") as pbar:
    test_distances = pairwise_distances(E_test_final, centers, metric="euclidean")
    test_df["cluster_after_pckmeans"] = np.argmin(test_distances, axis=1)
    pbar.update(1)

print("test cluster count:", test_df["cluster_after_pckmeans"].nunique())

train_metrics_before_correction = evaluate_clustering(
    df=df_train,
    pred_col="cluster_after_pckmeans",
    gold_col="relation"
)

test_metrics_before_correction = evaluate_clustering(
    df=test_df,
    pred_col="cluster_after_pckmeans",
    gold_col="relation"
)

print("\nTrain metrics before correction")
print(train_metrics_before_correction)

print("\nTest metrics before correction")
print(test_metrics_before_correction)


assign test clusters by nearest center:   0%|          | 0/1 [00:00<?, ?it/s]

test cluster count: 82


cluster_after_pckmeans B3 group index:   0%|          | 0/1900 [00:00<?, ?it/s]

cluster_after_pckmeans B3 score:   0%|          | 0/1900 [00:00<?, ?it/s]

Purity : 0.5563
ACC    : 0.3932
B3-F1  : 0.3352
V-F1   : 0.6058
ARI    : 0.2868


cluster_after_pckmeans B3 group index:   0%|          | 0/760 [00:00<?, ?it/s]

cluster_after_pckmeans B3 score:   0%|          | 0/760 [00:00<?, ?it/s]

Purity : 0.5868
ACC    : 0.4184
B3-F1  : 0.3923
V-F1   : 0.6650
ARI    : 0.3102

Train metrics before correction
{'Purity': 0.5563157894736842, 'ARI': 0.2867890002156228, 'V-hom': 0.6569743979425189, 'V-com': 0.5619480757639297, 'V-F1': 0.6057571448779032, 'B3-P': 0.4488818614711338, 'B3-R': 0.2674526315789477, 'B3-F1': 0.33519155166556514, 'ACC': 0.3931578947368421}

Test metrics before correction
{'Purity': 0.5868421052631579, 'ARI': 0.3102425119591357, 'V-hom': 0.7045284325928177, 'V-com': 0.6296987128554592, 'V-F1': 0.6650151717977766, 'B3-P': 0.4901192565960709, 'B3-R': 0.32697368421052764, 'B3-F1': 0.39225916913085657, 'ACC': 0.41842105263157897}


# 3단계. After clustering: low-confidence sample LLM Correction

In [15]:

def get_low_confidence_indices(distance_matrix, top_n):
    with tqdm(total=4, desc="select low-confidence samples") as pbar:
        sorted_dist = np.sort(distance_matrix, axis=1)
        pbar.update(1)

        nearest = sorted_dist[:, 0]
        second_nearest = sorted_dist[:, 1]
        pbar.update(1)

        margins = second_nearest - nearest
        pbar.update(1)

        top_n = min(top_n, len(margins))
        low_conf_indices = np.argsort(margins)[:top_n]
        pbar.update(1)

    return low_conf_indices, margins


low_conf_indices, test_margins = get_low_confidence_indices(
    distance_matrix=test_distances,
    top_n=num_low_confidence
)

test_df["cluster_margin"] = test_margins

print("num low-confidence samples:", len(low_conf_indices))
print("min margin:", test_margins.min())
print("max margin:", test_margins.max())


select low-confidence samples:   0%|          | 0/4 [00:00<?, ?it/s]

num low-confidence samples: 114
min margin: 0.00030505657
max margin: 0.8703153


In [16]:

def build_train_representatives(df_train, embeddings, centers, cluster_col, num_representatives=3):
    representatives = {}
    cluster_ids = sorted(df_train[cluster_col].unique())

    for cluster_id in tqdm(cluster_ids, desc="build train representatives"):
        cluster_indices = df_train.index[df_train[cluster_col] == cluster_id].tolist()

        if len(cluster_indices) == 0:
            representatives[int(cluster_id)] = []
            continue

        cluster_embeddings = embeddings[cluster_indices]
        center = centers[int(cluster_id)].reshape(1, -1)

        distances = pairwise_distances(cluster_embeddings, center, metric="euclidean").reshape(-1)
        order = np.argsort(distances)

        selected_indices = [cluster_indices[pos] for pos in order[:num_representatives]]

        reps = []
        for idx in selected_indices:
            row = df_train.loc[idx]
            reps.append({
                "idx": int(idx),
                "text": str(row["text"]) if pd.notna(row["text"]) else "",
                "subject": str(row["subject"]) if pd.notna(row["subject"]) else "",
                "object": str(row["object"]) if pd.notna(row["object"]) else "",
                "gpt4_phrase": str(row["gpt4_phrase"]) if pd.notna(row["gpt4_phrase"]) else ""
            })

        representatives[int(cluster_id)] = reps

    return representatives


cluster_representatives = build_train_representatives(
    df_train=df_train,
    embeddings=E_train_final,
    centers=centers,
    cluster_col="cluster_after_pckmeans",
    num_representatives=num_representatives
)


build train representatives:   0%|          | 0/109 [00:00<?, ?it/s]

In [17]:
def format_representatives(reps):
    lines = []

    for k, rep in enumerate(reps, start=1):
        lines.append(
            f"Representative {k}:\n"
            f"Sentence: {rep['text']}\n"
            f"Head entity: {rep['subject']}\n"
            f"Tail entity: {rep['object']}\n"
            f"Relational keyphrase: {rep['gpt4_phrase']}"
        )

    return "\n\n".join(lines)


def build_correction_prompt(target_row, current_cluster_id, candidate_cluster_ids, cluster_representatives):
    target_text = str(target_row["text"]) if pd.notna(target_row["text"]) else ""
    target_subj = str(target_row["subject"]) if pd.notna(target_row["subject"]) else ""
    target_obj = str(target_row["object"]) if pd.notna(target_row["object"]) else ""
    target_phrase = str(target_row["gpt4_phrase"]) if pd.notna(target_row["gpt4_phrase"]) else ""

    all_cluster_ids = [int(current_cluster_id)] + [
        int(cid) for cid in candidate_cluster_ids if int(cid) != int(current_cluster_id)
    ]

    all_cluster_ids = list(dict.fromkeys(all_cluster_ids))

    prompt_parts = []

    prompt_parts.append(
        "You are helping with relation-instance clustering.\n\n"
        "Given a target relation instance and several candidate clusters, "
        "choose the cluster that best expresses the same semantic relation between the head entity and the tail entity.\n\n"
        "If the current cluster is appropriate, return the current cluster ID.\n"
        "If another candidate cluster is more appropriate, return that cluster ID.\n"
        "Return only a JSON object in the following format:\n"
        '{"cluster_id": <integer>, "reason": "<short reason>"}'
    )

    prompt_parts.append(
        f"Target instance:\n"
        f"Sentence: {target_text}\n"
        f"Head entity: {target_subj}\n"
        f"Tail entity: {target_obj}\n"
        f"Relational keyphrase: {target_phrase}"
    )

    prompt_parts.append("Candidate clusters:")

    for cid in all_cluster_ids:
        reps = cluster_representatives.get(int(cid), [])
        rep_text = format_representatives(reps)

        prompt_parts.append(
            f"Cluster ID: {int(cid)}\n"
            f"{rep_text}"
        )

    return "\n\n".join(prompt_parts), all_cluster_ids


def parse_correction_output(output_text, valid_cluster_ids, fallback_cluster_id):
    text = output_text.strip()

    try:
        parsed = json.loads(text)
        cluster_id = int(parsed.get("cluster_id"))
        if cluster_id in valid_cluster_ids:
            return cluster_id
    except Exception:
        pass

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, dict):
            cluster_id = int(parsed.get("cluster_id"))
            if cluster_id in valid_cluster_ids:
                return cluster_id
    except Exception:
        pass

    for cid in valid_cluster_ids:
        if str(cid) in text:
            return int(cid)

    return int(fallback_cluster_id)


def call_llm_correction(
    target_row,
    current_cluster_id,
    candidate_cluster_ids,
    cluster_representatives,
    model_name="gpt-4.1-mini",
    max_retries=5,
    sleep_sec=2
):
    prompt, valid_cluster_ids = build_correction_prompt(
        target_row=target_row,
        current_cluster_id=current_cluster_id,
        candidate_cluster_ids=candidate_cluster_ids,
        cluster_representatives=cluster_representatives
    )

    for attempt in range(max_retries):
        try:
            result = client.responses.create(
                model=model_name,
                input=prompt,
                temperature=0
            )

            output_text = result.output_text

            selected_cluster = parse_correction_output(
                output_text=output_text,
                valid_cluster_ids=valid_cluster_ids,
                fallback_cluster_id=current_cluster_id
            )

            return selected_cluster, output_text

        except Exception as e:
            tqdm.write(f"[ERROR] correction LLM attempt={attempt + 1}, error={e}")
            time.sleep(sleep_sec)

    return int(current_cluster_id), "ERROR"


In [18]:

test_df["cluster_after_llm_correction"] = test_df["cluster_after_pckmeans"].copy()

correction_cache_path = os.path.join(
    save_dir,
    f"TACRED_test_llm_correction_{num_low_confidence}.csv"
)

if os.path.exists(correction_cache_path):
    correction_df = pd.read_csv(correction_cache_path)
    print("[LOAD]", correction_cache_path)

else:
    correction_records_by_order = {}
    changed_count = 0
    error_count = 0

    def run_correction_job(order, test_idx):
        test_idx = int(test_idx)
        target_row = test_df.iloc[test_idx]

        current_cluster_id = int(test_df.iloc[test_idx]["cluster_after_pckmeans"])

        nearest_cluster_order = np.argsort(test_distances[test_idx])
        candidate_cluster_ids = [
            int(cid)
            for cid in nearest_cluster_order[:num_candidate_clusters]
        ]

        if current_cluster_id not in candidate_cluster_ids:
            candidate_cluster_ids = [current_cluster_id] + candidate_cluster_ids
            candidate_cluster_ids = candidate_cluster_ids[:num_candidate_clusters]

        candidate_cluster_ids = list(dict.fromkeys(candidate_cluster_ids))

        selected_cluster, raw_output = call_llm_correction(
            target_row=target_row,
            current_cluster_id=current_cluster_id,
            candidate_cluster_ids=candidate_cluster_ids,
            cluster_representatives=cluster_representatives,
            model_name=llm_model
        )

        changed = int(current_cluster_id != int(selected_cluster))

        return order, {
            "test_index": test_idx,
            "current_cluster": current_cluster_id,
            "selected_cluster": int(selected_cluster),
            "candidate_cluster_ids": json.dumps(candidate_cluster_ids),
            "raw_output": raw_output,
            "changed": changed
        }

    low_conf_jobs = list(enumerate(low_conf_indices, start=1))

    with ThreadPoolExecutor(max_workers=max_parallel_api_calls) as executor:
        futures = [
            executor.submit(run_correction_job, order, test_idx)
            for order, test_idx in low_conf_jobs
        ]

        progress = tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"API low-confidence correction x{max_parallel_api_calls}"
        )
        for done_count, future in enumerate(progress, start=1):
            order, record = future.result()
            correction_records_by_order[order] = record

            changed_count += int(record["changed"])
            error_count += int(record["raw_output"] == "ERROR")

            test_idx = int(record["test_index"])
            selected_cluster = int(record["selected_cluster"])
            test_df.iloc[
                test_idx,
                test_df.columns.get_loc("cluster_after_llm_correction")
            ] = selected_cluster

            progress.set_postfix({"changed": changed_count, "error": error_count})

            if done_count % 20 == 0:
                ordered_records = [
                    correction_records_by_order[k]
                    for k in sorted(correction_records_by_order)
                ]
                correction_df = pd.DataFrame(ordered_records)
                correction_df.to_csv(correction_cache_path, index=False)
                test_df.to_csv(
                    os.path.join(save_dir, "TACRED_test_partial_after_llm_correction.csv"),
                    index=False
                )
                tqdm.write(f"[SAVE] correction {done_count}/{len(low_conf_indices)}")

    ordered_records = [
        correction_records_by_order[k]
        for k in sorted(correction_records_by_order)
    ]
    correction_df = pd.DataFrame(ordered_records)
    correction_df.to_csv(correction_cache_path, index=False)
    print("[DONE]", correction_cache_path)


if os.path.exists(correction_cache_path):
    for _, row in tqdm(
        correction_df.iterrows(),
        total=len(correction_df),
        desc="apply cached/applied corrections"
    ):
        test_idx = int(row["test_index"])
        selected_cluster = int(row["selected_cluster"])
        test_df.iloc[
            test_idx,
            test_df.columns.get_loc("cluster_after_llm_correction")
        ] = selected_cluster

print(correction_df["changed"].value_counts(dropna=False))


API low-confidence correction x20:   0%|          | 0/114 [00:00<?, ?it/s]

[SAVE] correction 20/114
[SAVE] correction 40/114
[SAVE] correction 60/114
[SAVE] correction 80/114
[SAVE] correction 100/114
[DONE] /workspace/KG-known0/ARR2026/Baselines/4_FewShotClus/output_TACRED/TACRED_test_llm_correction_114.csv


apply cached/applied corrections:   0%|          | 0/114 [00:00<?, ?it/s]

changed
1    67
0    47
Name: count, dtype: int64


# 최종 평가

In [19]:

test_df["cluster_after_llm_correction"] = test_df["cluster_after_pckmeans"].copy()

correction_cache_path = os.path.join(
    save_dir,
    f"TACRED_test_llm_correction_{num_low_confidence}.csv"
)

if os.path.exists(correction_cache_path):
    correction_df = pd.read_csv(correction_cache_path)
    print("[LOAD]", correction_cache_path)

else:
    correction_records_by_order = {}
    changed_count = 0
    error_count = 0

    def run_correction_job(order, test_idx):
        test_idx = int(test_idx)
        target_row = test_df.iloc[test_idx]

        current_cluster_id = int(test_df.iloc[test_idx]["cluster_after_pckmeans"])

        nearest_cluster_order = np.argsort(test_distances[test_idx])
        candidate_cluster_ids = [
            int(cid)
            for cid in nearest_cluster_order[:num_candidate_clusters]
        ]

        if current_cluster_id not in candidate_cluster_ids:
            candidate_cluster_ids = [current_cluster_id] + candidate_cluster_ids
            candidate_cluster_ids = candidate_cluster_ids[:num_candidate_clusters]

        candidate_cluster_ids = list(dict.fromkeys(candidate_cluster_ids))

        selected_cluster, raw_output = call_llm_correction(
            target_row=target_row,
            current_cluster_id=current_cluster_id,
            candidate_cluster_ids=candidate_cluster_ids,
            cluster_representatives=cluster_representatives,
            model_name=llm_model
        )

        changed = int(current_cluster_id != int(selected_cluster))

        return order, {
            "test_index": test_idx,
            "current_cluster": current_cluster_id,
            "selected_cluster": int(selected_cluster),
            "candidate_cluster_ids": json.dumps(candidate_cluster_ids),
            "raw_output": raw_output,
            "changed": changed
        }

    low_conf_jobs = list(enumerate(low_conf_indices, start=1))

    with ThreadPoolExecutor(max_workers=max_parallel_api_calls) as executor:
        futures = [
            executor.submit(run_correction_job, order, test_idx)
            for order, test_idx in low_conf_jobs
        ]

        progress = tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"API low-confidence correction x{max_parallel_api_calls}"
        )
        for done_count, future in enumerate(progress, start=1):
            order, record = future.result()
            correction_records_by_order[order] = record

            changed_count += int(record["changed"])
            error_count += int(record["raw_output"] == "ERROR")

            test_idx = int(record["test_index"])
            selected_cluster = int(record["selected_cluster"])
            test_df.iloc[
                test_idx,
                test_df.columns.get_loc("cluster_after_llm_correction")
            ] = selected_cluster

            progress.set_postfix({"changed": changed_count, "error": error_count})

            if done_count % 20 == 0:
                ordered_records = [
                    correction_records_by_order[k]
                    for k in sorted(correction_records_by_order)
                ]
                correction_df = pd.DataFrame(ordered_records)
                correction_df.to_csv(correction_cache_path, index=False)
                test_df.to_csv(
                    os.path.join(save_dir, "TACRED_test_partial_after_llm_correction.csv"),
                    index=False
                )
                tqdm.write(f"[SAVE] correction {done_count}/{len(low_conf_indices)}")

    ordered_records = [
        correction_records_by_order[k]
        for k in sorted(correction_records_by_order)
    ]
    correction_df = pd.DataFrame(ordered_records)
    correction_df.to_csv(correction_cache_path, index=False)
    print("[DONE]", correction_cache_path)


if os.path.exists(correction_cache_path):
    for _, row in tqdm(
        correction_df.iterrows(),
        total=len(correction_df),
        desc="apply cached/applied corrections"
    ):
        test_idx = int(row["test_index"])
        selected_cluster = int(row["selected_cluster"])
        test_df.iloc[
            test_idx,
            test_df.columns.get_loc("cluster_after_llm_correction")
        ] = selected_cluster

print(correction_df["changed"].value_counts(dropna=False))


[LOAD] /workspace/KG-known0/ARR2026/Baselines/4_FewShotClus/output_TACRED/TACRED_test_llm_correction_114.csv


apply cached/applied corrections:   0%|          | 0/114 [00:00<?, ?it/s]

changed
1    67
0    47
Name: count, dtype: int64


In [20]:


final_pred_col = "cluster_after_llm_correction"

if final_pred_col not in test_df.columns:
    raise ValueError(f"{final_pred_col} column does not exist in test_df.")

test_metrics_after_correction = evaluate_clustering(
    df=test_df,
    pred_col=final_pred_col,
    gold_col="relation"
)

print("\nTest metrics after LLM correction")
print(test_metrics_after_correction)

test_metrics_after_correction_df = pd.DataFrame([
    {
        "split": "test",
        "pred_col": final_pred_col,
        **test_metrics_after_correction
    }
])

test_metrics_after_correction_df

cluster_after_llm_correction B3 group index:   0%|          | 0/760 [00:00<?, ?it/s]

cluster_after_llm_correction B3 score:   0%|          | 0/760 [00:00<?, ?it/s]

Purity : 0.6092
ACC    : 0.4368
B3-F1  : 0.4115
V-F1   : 0.6821
ARI    : 0.3365

Test metrics after LLM correction
{'Purity': 0.6092105263157894, 'ARI': 0.3364828975693009, 'V-hom': 0.726624749376733, 'V-com': 0.6426689013064528, 'V-F1': 0.6820730222637494, 'B3-P': 0.5165365006776909, 'B3-R': 0.3419736842105273, 'B3-F1': 0.41150796641734205, 'ACC': 0.4368421052631579}


,split,pred_col,Purity,ARI,V-hom,V-com,V-F1,B3-P,B3-R,B3-F1,ACC
0,test,cluster_after_llm_correction,0.609211,0.336483,0.726625,0.642669,0.682073,0.516537,0.341974,0.411508,0.436842
